In [ ]:
# Import necessary libraries
import os
import re
import json
import logging
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer, util, losses, InputExample
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from bayes_opt import BayesianOptimization
from scipy.stats import friedmanchisquare, levene
import scikit_posthocs as sp
from tqdm.autonotebook import tqdm, trange
from langchain_ollama import OllamaLLM
from langchain.prompts import PromptTemplate

# Set plotting style to cyberpunk
import mplcyberpunk
plt.style.use("cyberpunk")

# Suppress the specific FutureWarning from huggingface_hub
warnings.filterwarnings("ignore", category=FutureWarning, module="huggingface_hub.file_download")


In [ ]:
# Load the probes dataset
probes_df = pd.read_csv("../data/probes.csv")
probes_df = probes_df.drop(['Description'], axis=1)

# Split "Compatible_Systems" column
probes_df['Compatible_Systems'] = probes_df['Compatible_Systems'].str.split(', ')

# Split "Applications" column
probes_df['Applications'] = probes_df['Applications'].str.split(', ')

# Load the systems dataset
systems_df = pd.read_csv("../data/systems.csv")

# Split "compatible_probes" column
systems_df['compatible_probes'] = systems_df['compatible_probes'].str.split(', ')

# Print the head of the dataframes
probes_df.head()

In [ ]:
systems_df.head()

## Generating document corpus

Here we convert our tabular data into string sentences and generate tags that correspond to the information it contains.

In [ ]:
# Function to generate probe descriptions and tags
def generate_probe_info(row):
    """
    This function takes a row from the probes_df DataFrame and generates lists of probe descriptions and tags.
    """
    descriptions = []
    tags = []

    # Manufacturer
    descriptions.append(f"The manufacturer of the {row['Probe_Model']} probe is {row['Manufacturer']}.")
    tags.append(frozenset(['probes', 'manufacturer', row['Manufacturer'], row['Probe_Model']]))

    # Cartridge Connection
    if row['Cartridge_Connection'] == 1:
        descriptions.append(f"The {row['Manufacturer']} {row['Probe_Model']} probe has a variant with a cartridge connection.")
        tags.append(frozenset(['probes', row['Manufacturer'], row['Probe_Model'], 'connection']))

    # Compatible Systems
    compatible_systems = ', '.join(row['Compatible_Systems'])
    descriptions.append(f"The {row['Manufacturer']} {row['Probe_Model']} probe is compatible with the following systems: {compatible_systems}.")
    tags.append(frozenset(['probes', row['Manufacturer'], row['Probe_Model'], 'compatibility', compatible_systems]))

    # Array Type
    descriptions.append(f"The {row['Manufacturer']} {row['Probe_Model']} is a {row['Array_Type']} array type probe.")
    tags.append(frozenset(['probes', row['Manufacturer'], row['Probe_Model'], 'array type']))

    # Applications
    applications = ', '.join(row['Applications'])
    descriptions.append(f"The {row['Manufacturer']} {row['Probe_Model']} probe is suitable for the following applications: {applications}.")
    tags.append(frozenset(['probes', row['Manufacturer'], row['Probe_Model'], "applications", applications]))

    # Stock
    if row['Stock'] > 0:
        descriptions.append(f"The {row['Manufacturer']} {row['Probe_Model']} probe is currently in stock and available for sale.")
    else:
        descriptions.append(f"The {row['Manufacturer']} {row['Probe_Model']} probe is currently out of stock or not available for sale.")
    tags.append(frozenset(['probes', row['Manufacturer'], row['Probe_Model'], 'stock']))

    # Frequency Range
    descriptions.append(f"The frequency range of the {row['Manufacturer']} {row['Probe_Model']} probe is {row['Min_Frequency']} - {row['Max_Frequency']} MHz.")
    tags.append(frozenset(['probes', row['Manufacturer'], row['Probe_Model'], 'frequency']))

    return descriptions, tags

# Function to generate system descriptions and tags
def generate_system_description(row):
    """
    This function takes a row from the systems_df DataFrame and generates lists of system descriptions and tags.
    """
    descriptions = []
    tags = []

    # System manufacturer
    description = f"The {row['ultrasound_system']} ultrasound system is made by {row['manufacturer']}."
    descriptions.append(description)
    tags.append(frozenset([row['manufacturer'], row['ultrasound_system'], 'manufacturer', 'systems']))

    # System compatible probes
    compatible_probes = ', '.join(row['compatible_probes'])
    description = f"The {row['manufacturer']} {row['ultrasound_system']} ultrasound system is compatible with the following {row['manufacturer']} probes: {compatible_probes}."
    descriptions.append(description)
    tags.append(frozenset([row['manufacturer'], row['ultrasound_system'], 'compatibility', 'systems', compatible_probes]))

    return descriptions, tags

# Apply the probe function and directly unpack into separate lists
probe_descriptions, probe_tags = zip(*probes_df.apply(generate_probe_info, axis=1))

# Apply the system function and directly unpack into separate lists
system_descriptions, system_tags = zip(*systems_df.apply(generate_system_description, axis=1))

# Flatten the lists
all_contents = [item for sublist in (probe_descriptions + system_descriptions) for item in sublist]
all_tags = [item for sublist in (probe_tags + system_tags) for item in sublist]

# Create a DataFrame from these lists
documents = pd.DataFrame({
    'content': all_contents,
    'tags': all_tags
})

documents.head()

## Generating queries for the documents using Ollama (llama3.2)

We define a prompt and include examples of what questions to ask for each document type. We also instruct the prompt to structure its output using numbers to separate questions so each question can be extracted using regex. 



In [ ]:
# Load the Ollama LLM
llm = OllamaLLM(model="llama3.2")

prompt_template = PromptTemplate(
    input_variables=["document"],
    template="""
Generate three concise questions that can be answered using the following information, similar to the example provided. Provide only the questions, numbered as follows:

1. [insert question here]
2. [insert question here]
3. [insert question here]

Example 1:
Information: The manufacturer of the C3 probe is ATL.
Questions:
1. Who is the manufacturer of the C3 probe?
2. Who makes the C3 transducer?
3. Is the C3 probe made by ATL?

Example 2:
Information: The G.E. RIC5-9D probe is compatible with the following systems: Voluson E6, Voluson E8, Voluson E10, LOGIQ S7, LOGIQ S8, Vivid E95, LOGIQ E9, LOGIQ E10.
Questions:
1. What systems are compatible with the G.E. RIC5-9D probe?
2. Does the G.E. RIC5-9D work with the Voluson E6 system?
3. Is the G.E. RIC5-9D transducer compatible with the LOGIQ S7 system?

Example 3:
Information: The Siemens Acuson 15L8W probe has a variant with a cartridge connection.
Questions:
1. Does the Siemens Acuson 15L8W probe have a cartridge connection?
2. What kind of connector does the Siemens Acuson 15L8W transducer use?
3. Does the Siemens Acuson 15L8W probe use a cartridge connector?

Example 4:
Information: The Philips C8-4v is a convex array type probe.
Questions:
1. What array type is the Philips C8-4v?
2. What type of transducer is the Philips C8-4v?
3. Is the Philips C8-4v a convex array probe?

Example 5:
Information: The ATL P6-3 probe is suitable for the following applications: cardiac, vascular, general.
Questions:
1. What can the ATL P6-3 probe be used for?
2. What are the applications of the ATL P6-3 probe?
3. Can the P6-3 probe be used for cardiac applications?

Example 6:
Information: The G.E. RAB2-5L probe is currently in stock and available for sale.
Questions:
1. Do you have any RAB2-5L probes by G.E. for sale?
2. Is the G.E. RAB2-5L probe in stock?
3. Are there any RAB2-5L transducers for sale?

Example 7:
Information: The frequency range of the ATL L10-5 probe is 5.0 - 10.0 MHz.
Questions:
1. What is the frequency range of the ATL L10-5 probe?
2. What is the maximum frequency range of the ATL L10-5 transducer?
3. What is the minimum frequency range of the ATL L10-5 probe?

Information:
{document}
"""
)

def extract_questions(response):
    pattern = r'\d+\.\s(.+?\?)'
    questions = re.findall(pattern, response)
    return questions

def generate_questions(text_chunk):
    prompt = prompt_template.format(document=text_chunk)
    output = llm(prompt)
    questions = extract_questions(output)
    return questions

# Generate questions for each document
tqdm.pandas(desc="Generating questions")
documents["questions"] = documents["content"].progress_apply(generate_questions)

In [ ]:
# Create a new DataFrame by exploding the 'questions' column
questions_df = documents.explode('questions')
questions_df = questions_df.reset_index(drop=True)
questions_df = questions_df.rename(columns={'questions': 'question'})
questions_df.to_csv("../data/questions_docs_tags.csv")
questions_df

## Filtering out the questions that are not relevant using llama3 again

Any document-query pairs that are marked irrelevant are exported, manually reviewed, and re-scored if neccessary before being joined back into the dataset.


In [ ]:
# Define the prompt template for grading questions
grading_prompt_template = PromptTemplate(
    input_variables=["content", "question"],
    template="""
    Given the content: "{content}"
    Can the following question be sufficiently answered?
    Question: "{question}"
    Your response should only consist of one number, either a 0 (meaning No) or 1 (meaning Yes).
    
    Example 1:
    Content: "The manufacturer of the C3 probe is ATL."
    Question: "Who is the manufacturer of the C3 probe?"
    Response: 1

    Example 2:
    Content: "Who manufactures the EPIQ 7 ultrasound system?"
    Question: "The Siemens Acuson 12L3 probe is compatible with the following systems: Juniper."
    Response: 0
    """
)

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

# Precompile the regular expression for efficiency
score_pattern = re.compile(r'\b[01]\b')

def grade_question(content, question):
    prompt = grading_prompt_template.format(content=content, question=question)
    response = llm(prompt)
    # Use the precompiled regular expression to find numbers in the response
    matches = score_pattern.findall(response)
    if matches:
        # Assuming the first match is the relevant score
        return int(matches[0])
    else:
        # Log the issue and return a default value
        logging.warning(f"No valid score found in response: {response}")
        return None  # or set a default value, or handle it in another way


# Apply the grade_question function to each row to create a new 'score' column
tqdm.pandas(desc="Scoring Questions")
questions_df['score'] = questions_df.progress_apply(lambda row: grade_question(row['content'], row['question']), axis=1)
print(questions_df.head())

In [ ]:
filtered_df = questions_df[questions_df['score'].isin([0, None])]
filtered_df

Exporting for manual review

In [ ]:
#filtered_df.to_csv("../data/filtered_questions_for_correction.csv")

Re-importing manually reviewed pairs

In [ ]:
#corrected_questions_df = pd.read_csv("../data/corrected_questions.csv")

# Drop the old rows from questions_df
#fine_tuning_dataset = questions_df.drop(filtered_df.index)

# Append the corrected rows to the original DataFrame
#fine_tuning_dataset = pd.concat([fine_tuning_dataset, corrected_questions_df], ignore_index=True)
#fine_tuning_dataset = fine_tuning_dataset[fine_tuning_dataset['score'] == 1].drop(columns=['score'])
#fine_tuning_dataset



In [ ]:
# Generate unique IDs for queries and documents based on their content
#document_ids = {doc: idx for idx, doc in enumerate(fine_tuning_dataset['content'].unique())}
#fine_tuning_dataset['doc_id'] = fine_tuning_dataset['content'].map(document_ids)
#fine_tuning_dataset['query_id'] = range(len(fine_tuning_dataset))

#fine_tuning_dataset.to_pickle('../data/fine_tuning_dataset.pkl')

## Start here to import cleaned question-document dataset

In [ ]:
# Loading from Pickle
fine_tuning_dataset = pd.read_pickle('../data/fine_tuning_dataset.pkl')
fine_tuning_dataset = fine_tuning_dataset[fine_tuning_dataset['labels'] == 1.0]
fine_tuning_dataset

## Evaluating pre-trained embedding models

In [ ]:
# Merging back the tags with the fine-tuning dataset
tags_df = pd.read_csv("../data/questions_docs_tags.csv")
tags_df = tags_df.drop(columns=['Unnamed: 0'])
tags_df_unique = tags_df.groupby('content')['tags'].apply(lambda x: ','.join(x)).reset_index()
merged_df = pd.merge(fine_tuning_dataset, tags_df_unique[['content', 'tags']], on='content', how='left')
merged_df = merged_df[merged_df['labels'] == 1.0]
merged_df


In [ ]:
# Load the models
models = {
    'multi-qa-mpnet-base-dot-v1': SentenceTransformer('multi-qa-mpnet-base-dot-v1'),
    'all-mpnet-base-v2': SentenceTransformer('all-mpnet-base-v2'),
    'multi-qa-distilbert-cos-v1': SentenceTransformer('multi-qa-distilbert-cos-v1')
}

# Generate embeddings for the content and questions
embeddings = {}
for model_name, model in models.items():
    embeddings[model_name] = {
        'content': model.encode(merged_df['content'].tolist(), convert_to_tensor=True, show_progress_bar=False),
        'question': model.encode(merged_df['question'].tolist(), convert_to_tensor=True, show_progress_bar=False)
    }



In [ ]:
def compute_rr(embeddings, ranking_method):
    results_df = merged_df.copy()
    results_df['absolute_rank'] = 0
    results_df['rr'] = 0.0

    for i, question in enumerate(merged_df['question']):
        relevant_doc = merged_df['content'].iloc[i]
        query_emb = embeddings['question'][i].unsqueeze(0)  # Add batch dimension

        if ranking_method == 'dot_score':
            scores = util.dot_score(query_emb, embeddings['content'])[0].cpu().tolist()
        elif ranking_method == 'cos_sim':
            scores = util.cos_sim(query_emb, embeddings['content'])[0].cpu().tolist()

        # Combine docs & scores
        doc_score_pairs = list(zip(merged_df['content'], scores))

        # Sort by decreasing score
        doc_score_pairs = sorted(doc_score_pairs, key=lambda x: x[1], reverse=True)

        # Find the rank of the first relevant document
        matching_indices = [idx for idx, (doc, score) in enumerate(doc_score_pairs) if doc == relevant_doc]
        if len(matching_indices) > 0:
            rank = matching_indices[0] + 1
        else:
            rank = len(doc_score_pairs) + 1  # Relevant document not found

        # Calculate reciprocal rank
        rr = 1 / rank

        # Update the corresponding row in results_df
        results_df.at[i, 'absolute_rank'] = rank
        results_df.at[i, 'rr'] = rr

    return results_df

# Compute RR for each model and ranking method
results = {
    'multi-qa-mpnet-base-dot-v1': compute_rr(embeddings['multi-qa-mpnet-base-dot-v1'], 'dot_score'),
    'all-mpnet-base-v2': compute_rr(embeddings['all-mpnet-base-v2'], 'dot_score'),
    'multi-qa-distilbert-cos-v1': compute_rr(embeddings['multi-qa-distilbert-cos-v1'], 'cos_sim')
}

In [ ]:
# Define the order of models
model_order = ['all-mpnet-base-v2', 'multi-qa-mpnet-base-dot-v1', 'multi-qa-distilbert-cos-v1']

# Filter RR values for different 'probes' and 'systems' question types
probe_types = ['manufacturer', 'compatibility', 'array type', 'applications', 'stock', 'frequency']
system_types = ['manufacturer', 'compatibility']

results_rr_by_type = {key: {ptype: df[df['tags'].apply(lambda x: 'probes' in x and ptype in x)]['rr'].dropna() for ptype in probe_types} for key, df in results.items()}

# Add 'systems' question types to the results
for key, df in results.items():
    results_rr_by_type[key].update({f'systems_{stype}': df[df['tags'].apply(lambda x: 'systems' in x and stype in x)]['rr'].dropna() for stype in system_types})

# Define the complete list of question types
all_types = probe_types + [f'systems_{stype}' for stype in system_types]
labels = [ptype.capitalize() for ptype in probe_types] + [f'Sys {stype.capitalize()}' for stype in system_types]

# Create the boxplot
fig, ax = plt.subplots(len(model_order), 1, figsize=(6, 7))  # Adjust the figure size

for i, model_key in enumerate(model_order):
    rr_by_type = results_rr_by_type[model_key]
    data = [rr_by_type[ptype] for ptype in all_types if ptype in rr_by_type]
    box = ax[i].boxplot(data, vert=True, patch_artist=True, medianprops=dict(color='yellow', alpha=0.5), boxprops=dict(facecolor='cyan', alpha=0.3, edgecolor='cyan'), capprops=dict(color='cyan', alpha=0.3), whiskerprops=dict(color='yellow'), flierprops=dict(markeredgecolor='yellow', markerfacecolor='yellow', alpha=0.3))
    ax[i].set_xticklabels(labels, rotation=45, fontsize=8)  # Reduced font size for x-tick labels
    ax[i].set_ylabel('RR', fontsize=10)  # Reduced font size for y-label
    ax[i].set_title(f'{model_key} Reciprocal Rank Distribution by Question Type', fontsize=12)  # Reduced font size for title

    mplcyberpunk.make_lines_glow(ax[i])

plt.tight_layout()
plt.show()

Levene's to test variance homogeneity across models and whether to use a parametric or non-parametric test.

- Null Hypothesis (H0): The variances of MRR scores are equal across the different models.
- Alternative Hypothesis (H1): The variances of MRR scores are not equal across the different models.

In [ ]:
# Prepare data for Levene's test by model
data_by_model = {model_key: [] for model_key in model_order}
for model_key in model_order:
    for ptype in all_types:
        if ptype in results_rr_by_type[model_key]:
            data_by_model[model_key].extend(results_rr_by_type[model_key][ptype])

# Perform Levene's test by model
levene_result_by_model = levene(*data_by_model.values())

# Check the result
print(f"Levene's test by model p-value = {levene_result_by_model.pvalue}")

The variances of MRR scores are not equal across the different models so we will use Friedman Test (non parametric repeated measure one-way ANOVA) and Nemenyi post hoc test

In [ ]:
# Calculate means and standard errors
mean_rr_by_type = {}
se_rr_by_type = {}
friedman_results = {}

for ptype in all_types:
    mean_rr_by_type[ptype] = []
    se_rr_by_type[ptype] = []
    friedman_data = []
    for model_name in model_order:  # Use the defined model order
        rr_by_type = results_rr_by_type[model_name][ptype]
        mean_rr = np.mean(rr_by_type)
        se_rr = np.std(rr_by_type) / np.sqrt(len(rr_by_type))  # Standard Error of the mean
        mean_rr_by_type[ptype].append(mean_rr)
        se_rr_by_type[ptype].append(se_rr)
        friedman_data.append(rr_by_type)

    # Perform Friedman Test
    friedman_result = friedmanchisquare(*friedman_data)
    friedman_results[ptype] = friedman_result

    # If Friedman Test is significant, perform Nemenyi post hoc test
    if friedman_result.pvalue < 0.05:
        # Combine data into a DataFrame for the Nemenyi test
        data = {model_name: rr_by_type for model_name, rr_by_type in zip(model_order, friedman_data)}
        data_df = pd.DataFrame(data)
        nemenyi_result = sp.posthoc_nemenyi_friedman(data_df)

# Colors for different models
colors = ['cyan', 'magenta', 'yellow']

# New labels for the models
new_model_order = ['all-mpnet-base', 'QA-mpnet', 'QA-DistilBERT']

for ptype in all_types:
    # Show plot only if Friedman test indicates significant difference
    if friedman_results[ptype].pvalue < 0.05:
        # Create a new figure for each question type
        fig, ax = plt.subplots(figsize=(5, 2.75)) 

        # Data for plotting
        means = mean_rr_by_type[ptype]
        errors = se_rr_by_type[ptype]
        x = np.arange(len(means))

        # Creating bar chart
        bars = ax.bar(x, means, yerr=errors, color=colors, capsize=7, alpha=0.3, linewidth=2, ecolor='white')
        for bar, color in zip(bars, colors):
            bar.set_edgecolor(color)
        ax.set_xticks(x)
        ax.set_xticklabels(new_model_order, fontsize=8)  # Set new model order labels
        ax.set_title(f'MRR and SE for "{ptype}" type questions', fontsize=10)
        ax.set_ylim(0, 1)
        ax.set_ylabel('Mean Reciprocal Rank', fontsize=9)
        ax.set_yticks(np.linspace(0, 1, 6))

        mplcyberpunk.make_lines_glow(ax, alpha_line=0.6)

        plt.tight_layout(rect=[0, 0.1, 1, 1])  # Adjust the bottom margin
        plt.savefig(f'../images/bar_chart_{ptype}.png', dpi=90, bbox_inches='tight')
        plt.show()

## Fine-tuning

In [ ]:
# Remove duplicates and reset index
unique_corpus = fine_tuning_dataset.drop_duplicates(subset=['content']).reset_index(drop=True)

# Create document IDs
document_ids = {doc: idx for idx, doc in enumerate(unique_corpus['content'])}

# Create queries and query-document IDs
queries = {idx: row['question'] for idx, row in fine_tuning_dataset.iterrows()}
query_doc_ids = {idx: document_ids[row['content']] for idx, row in fine_tuning_dataset.iterrows() if row['labels'] == 1.0}

# Split the data into training, validation, and test sets
train_queries, temp_queries = train_test_split(list(queries.keys()), test_size=0.3, random_state=42)
val_queries, test_queries = train_test_split(temp_queries, test_size=0.5, random_state=42)

# Create relevant documents for validation and test sets
relevant_docs_val = {q_id: {query_doc_ids[q_id]} for q_id in val_queries if q_id in query_doc_ids}
relevant_docs_test = {q_id: {query_doc_ids[q_id]} for q_id in test_queries if q_id in query_doc_ids}

# Create corpus dictionary
corpus = {doc_id: doc for doc, doc_id in document_ids.items()}


In [ ]:
class QADataset(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]

def get_content(query_id):
    doc_id = query_doc_ids.get(query_id, None)
    if doc_id is not None:
        return corpus[doc_id]
    else:
        return ""

train_examples = [InputExample(texts=[queries[q_id], get_content(q_id)]) for q_id in train_queries]
val_examples = [InputExample(texts=[queries[q_id], get_content(q_id)]) for q_id in val_queries]
test_examples = [InputExample(texts=[queries[q_id], get_content(q_id)]) for q_id in test_queries]

train_dataset = QADataset(train_examples)
val_dataset = QADataset(val_examples)
test_dataset = QADataset(test_examples)


In [ ]:
class LoggingInformationRetrievalEvaluator(InformationRetrievalEvaluator):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.logger = logging.getLogger('training_logger')
        self.last_logged_epoch = -1

    def __call__(self, model, output_path=None, epoch=-1, steps=-1):
        if epoch != self.last_logged_epoch:
            results = super().__call__(model, output_path, epoch, steps)
            log_message = f"Epoch: {epoch}, Steps: {steps}\n"
            log_message += f"Information Retrieval Evaluation of the model on the {self.name} dataset:\n"
            log_message += f"Queries: {len(self.queries)}\nCorpus: {len(self.corpus)}\n\n"
            
            # Log only the key metrics
            key_metrics = [
                'validation-ir-eval_cosine_accuracy@1',
                'validation-ir-eval_cosine_accuracy@5',
                'validation-ir-eval_cosine_recall@1',
                'validation-ir-eval_cosine_recall@5',
                'validation-ir-eval_cosine_ndcg@10',
                'validation-ir-eval_cosine_mrr@10',
                'validation-ir-eval_cosine_map@100'
            ]
            for metric in key_metrics:
                if metric in results:
                    log_message += f"{metric}: {results[metric]}\n"
            
            self.logger.info(log_message)
            self.last_logged_epoch = epoch
            return results
        else:
            return super().__call__(model, output_path, epoch, steps)

# Setup logging configuration
def setup_logging(log_file_path):
    logger = logging.getLogger('training_logger')
    logger.setLevel(logging.INFO)
    fh = logging.FileHandler(log_file_path)
    fh.setLevel(logging.INFO)
    formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    fh.setFormatter(formatter)
    logger.addHandler(fh)
    return logger, fh

## Bayesian Optimization for Hyperparameter Tuning

In [ ]:
# Function to remove and close all handlers from the logger
def close_logger(logger):
    handlers = logger.handlers[:]
    for handler in handlers:
        handler.close()
        logger.removeHandler(handler)

# Delete the existing training log file if it exists
log_file_path = '../models/bayes_opt/training.log'
if os.path.exists(log_file_path):
    close_logger(logging.getLogger('training_logger'))
    os.remove(log_file_path)

# Define the objective function
def objective(per_gpu_batch_size, weight_decay, learning_rate, warmup_steps, num_epochs):
    per_gpu_batch_size = int(per_gpu_batch_size)
    warmup_steps = int(warmup_steps)
    num_epochs = int(num_epochs)

    # Load the model
    model = SentenceTransformer('multi-qa-distilbert-cos-v1')

    # Define the loss function
    loss = losses.MultipleNegativesRankingLoss(model=model)

    # Setup logging
    logger, fh = setup_logging('../models/bayes_opt/training.log')

    # Instantiate the custom evaluators
    ir_evaluator_val = LoggingInformationRetrievalEvaluator(
        queries={q_id: queries[q_id] for q_id in val_queries},
        corpus=corpus,
        relevant_docs=relevant_docs_val,
        name="validation-ir-eval"
    )

    # Create DataLoader with the suggested batch size
    train_dataloader = DataLoader(train_dataset, batch_size=per_gpu_batch_size, shuffle=True)

    # Train the model with the evaluator
    model.fit(
        train_objectives=[(train_dataloader, loss)],
        evaluator=ir_evaluator_val,
        epochs=num_epochs,
        warmup_steps=warmup_steps,
        evaluation_steps=500,
        save_best_model=False,  # Do not save the model
        show_progress_bar=True,
        optimizer_params={'lr': learning_rate, 'weight_decay': weight_decay}
    )

    # Evaluate the model on the validation set
    results = ir_evaluator_val(model)
    
    # Close the logger handlers to release the file
    close_logger(logger)
    
    # Return the metric to be optimized, defaulting to 0 if 'validation-ir-eval_cosine_mrr@10' is not found
    return results.get('validation-ir-eval_cosine_mrr@10', 0)

# Define the search space
pbounds = {
    'per_gpu_batch_size': (16, 64),
    'weight_decay': (0, 0.3),
    'learning_rate': (1e-5, 5e-5),
    'warmup_steps': (0, 500),
    'num_epochs': (2, 5)
}

# Create a BayesianOptimization object
optimizer = BayesianOptimization(
    f=objective,
    pbounds=pbounds,
    random_state=42,
    verbose=2
)

# Run the optimization with reduced number of trials
optimizer.maximize(init_points=5, n_iter=15)

# Print the best hyperparameters
best_hyperparameters = optimizer.max['params']
print("Best hyperparameters: ", best_hyperparameters)

# Save the best hyperparameters to a file
with open('../models/bayes_opt/best_hyperparameters.json', 'w') as f:
    json.dump(best_hyperparameters, f)

In [ ]:
# Load best hyperparameters 
file_path = '../models/bayes_opt/best_hyperparameters.json'
with open(file_path, 'r') as file:
    best_hyperparameters = json.load(file)

## Fine-tuning with best hyperparameters

In [ ]:
# Use the best hyperparameter for batch size
batch_size = int(best_hyperparameters['per_gpu_batch_size'])

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Train the final model with the best hyperparameters
def train_final_model(best_hyperparameters):
    per_gpu_batch_size = int(best_hyperparameters['per_gpu_batch_size'])
    weight_decay = best_hyperparameters['weight_decay']
    learning_rate = best_hyperparameters['learning_rate']
    warmup_steps = int(best_hyperparameters['warmup_steps'])
    num_epochs = int(best_hyperparameters['num_epochs'])

    # Load the model
    model = SentenceTransformer('multi-qa-distilbert-cos-v1')

    # Define the loss function
    loss = losses.MultipleNegativesRankingLoss(model=model)

    # Setup logging
    logger, fh = setup_logging('../models/best-IR-model/final_training.log')

    # Instantiate the custom evaluators
    ir_evaluator_val = LoggingInformationRetrievalEvaluator(
        queries={q_id: queries[q_id] for q_id in val_queries},
        corpus=corpus,
        relevant_docs=relevant_docs_val,
        name="validation-ir-eval"
    )

    # Create DataLoader with the suggested batch size
    train_dataloader = DataLoader(train_dataset, batch_size=per_gpu_batch_size, shuffle=True)

    # Train the model with the evaluator
    model.fit(
        train_objectives=[(train_dataloader, loss)],
        evaluator=ir_evaluator_val,
        epochs=num_epochs,
        warmup_steps=warmup_steps,
        evaluation_steps=500,
        save_best_model=True,  # Save the best model
        show_progress_bar=True,
        optimizer_params={'lr': learning_rate, 'weight_decay': weight_decay}
    )

    # Save the trained model
    model_save_path = '../models/best-IR-model'
    model.save(model_save_path)

    # Close the logger handlers to release the file
    close_logger(logger)

    print(f"Final model saved to {model_save_path}")

# Train the final model with the best hyperparameters
train_final_model(best_hyperparameters)


In [ ]:
# Read the log file
with open('../models/best-IR-model/final_training.log', 'r') as f:
    log_contents = f.read()

# Extract metrics using regular expressions
epoch_pattern = r'Epoch: (\d+\.\d+)'
metric_pattern = r'(validation-ir-eval_cosine_(\w+))@(\d+): (\d+\.\d+)'

epochs = re.findall(epoch_pattern, log_contents)
metrics = re.findall(metric_pattern, log_contents)

# Create a dictionary to store the metric values for each epoch, grouped by metric type and top_k
metric_values = {}
for metric_name, metric_type, top_k, metric_value in metrics:
    metric_values.setdefault(metric_type, {}).setdefault(top_k, []).append(float(metric_value))

# Define colors and line styles for each metric and top_k value
colors = {
    'accuracy@1': 'cyan', 'accuracy@5': 'cyan',
    'recall@1': 'yellow', 'recall@5': 'yellow',
    'mrr@10': 'magenta', 'ndcg@10': 'green'
}
line_styles = {
    'accuracy@1': '-', 'accuracy@5': '--',
    'recall@1': '-', 'recall@5': '--',
    'mrr@10': '-', 'ndcg@10': '--'
}

def plot_metrics(metric_types, title, filename, y_lim=(0.7, 1.05), y_ticks=np.arange(0.7, 1.05, 0.1), dpi=100):
    # Adjust font sizes
    plt.rcParams.update({'font.size': 8})  # Adjusting the global font size smaller for smaller plots

    # Convert pixel dimensions to inches for figsize
    width_in_inches = 400 / dpi
    height_in_inches = 250 / dpi

    plt.figure(figsize=(width_in_inches, height_in_inches))
    for metric_type in metric_types:
        for top_k, values in metric_values[metric_type].items():
            plt.plot(range(1, len(epochs) + 1), values, color=colors[f'{metric_type}@{top_k}'],
                     linestyle=line_styles[f'{metric_type}@{top_k}'], label=f'{metric_type}@{top_k}', alpha=0.7)
    plt.title(title, pad=10)
    plt.ylabel('Metric Value')
    plt.ylim(*y_lim)
    plt.xlim(1, len(epochs))
    plt.yticks(y_ticks)
    plt.grid(True)
    mplcyberpunk.add_glow_effects()
    handles, labels = plt.gca().get_legend_handles_labels()
    unique_labels = list(set(labels))
    unique_handles = [handles[labels.index(label)] for label in unique_labels]
    plt.legend(unique_handles, unique_labels, bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0)
    plt.xlabel('Epoch')
    plt.xticks(range(1, len(epochs) + 1, 1))
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig(f'../images/{filename}', dpi=dpi)  
    plt.show()

# Plot accuracy, recall, NDCG and MRR
plot_metrics(metric_types=['accuracy'], title='Accuracy', filename='val_accuracy.png')
plot_metrics(metric_types=['recall'], title='Recall', filename='val_recall.png')
plot_metrics(metric_types=['ndcg', 'mrr'], title='NDCG and MRR', filename='val_ndcg_mrr.png')



In [ ]:
# Load your best model checkpoint
best_model_path = '../models/best-IR-model'
best_model = SentenceTransformer(best_model_path)

# Load the base pre-fine-tuning model and mpnet models
base_model = SentenceTransformer('multi-qa-distilbert-cos-v1')
mpnet_model = SentenceTransformer('multi-qa-mpnet-base-dot-v1')
all_mpnet_model = SentenceTransformer('all-mpnet-base-v2')

In [ ]:
# Create the evaluator
ir_evaluator = InformationRetrievalEvaluator(
    queries={q_id: queries[q_id] for q_id in test_queries},
    corpus=corpus,
    relevant_docs=relevant_docs_test,
    name="test-ir-eval"
)

# Evaluate the models
results_best_model = ir_evaluator(best_model)
results_base_model = ir_evaluator(base_model)
results_mpnet_model = ir_evaluator(mpnet_model)
results_all_mpnet_model = ir_evaluator(all_mpnet_model)

In [ ]:
# Define the metrics and top_k values
metric_names = ['accuracy', 'precision', 'recall', 'ndcg', 'mrr']
top_k_values = {
    'accuracy': [1, 5],
    'precision': [1],
    'recall': [1, 5],
    'ndcg': [10],
    'mrr': [10]
}

# Create a list of model names and their colors
model_names = ['QA-distilbert (fine-tuned)', 'QA-distilbert (base model)', 'QA-mpnet', 'All-mpnet-base-v2']
model_colors = ['cyan', 'magenta', 'yellow', 'green']

# Function to plot metrics
def plot_metrics(metric_name, top_k_values, model_names, model_colors, dpi=225):
    x = np.arange(len(top_k_values[metric_name]))  # Set the x-axis positions for the bars
    width = 0.2  # Width of the bars

    # Convert pixel dimensions to inches for figsize
    width_in_inches = 1000 / dpi
    height_in_inches = 350 / dpi

    fig, ax = plt.subplots(figsize=(width_in_inches, height_in_inches))

    for j, top_k in enumerate(top_k_values[metric_name]):
        metric_key = f'test-ir-eval_cosine_{metric_name}@{top_k}'
        
        # Extract the metric values for each model
        metric_values = [
            results_best_model.get(metric_key, 0),
            results_base_model.get(metric_key, 0),
            results_mpnet_model.get(metric_key, 0),
            results_all_mpnet_model.get(metric_key, 0)
        ]

        # Create the bar chart for the current metric
        for i, (model_name, color) in enumerate(zip(model_names, model_colors)):
            ax.bar(x[j] + (i - 1.5) * width, metric_values[i], width, color=color, label=model_name, alpha=0.3)

    # Set the x-axis labels and tick positions
    ax.set_xticks(x)
    ax.set_xticklabels([f'k={top_k}' for top_k in top_k_values[metric_name]])

    # Set the title and labels for the current subplot
    title_map = {'ndcg': 'NDCG', 'mrr': 'MRR'}
    metric_title = title_map.get(metric_name, metric_name.capitalize())
    ax.set_title(f'{metric_title} @ top-k documents retrieved')
    ax.set_ylabel('Score')

    # Set y-axis limits and ticks
    ax.set_ylim(0, 1)
    ax.set_yticks(np.arange(0, 1.1, 0.2))

    # Remove duplicate legend entries and show only model names in the correct order
    handles, labels = ax.get_legend_handles_labels()
    unique_labels, unique_handles = [], []
    for model_name in model_names:
        if model_name in labels:
            unique_labels.append(model_name)
            unique_handles.append(handles[labels.index(model_name)])
    ax.legend(unique_handles, unique_labels, bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0)

    # Adjust the layout and save the plot
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig(f'../images/{metric_name}_test_plot.png', dpi=dpi)
    plt.show()

# Iterate over each metric and plot
for metric_name in metric_names:
    plot_metrics(metric_name, top_k_values, model_names, model_colors)

In [ ]:
# Combine results into a DataFrame
data = {
    'QA-distilbert (fine-tuned)': [
        results_best_model['test-ir-eval_cosine_accuracy@1'],
        results_best_model['test-ir-eval_cosine_accuracy@5'],
        results_best_model['test-ir-eval_cosine_precision@1'],
        results_best_model['test-ir-eval_cosine_recall@1'],
        results_best_model['test-ir-eval_cosine_recall@5'],
        results_best_model['test-ir-eval_cosine_ndcg@10'],
        results_best_model['test-ir-eval_cosine_mrr@10']
    ],
    'QA-distilbert (base model)': [
        results_base_model['test-ir-eval_cosine_accuracy@1'],
        results_base_model['test-ir-eval_cosine_accuracy@5'],
        results_base_model['test-ir-eval_cosine_precision@1'],
        results_base_model['test-ir-eval_cosine_recall@1'],
        results_base_model['test-ir-eval_cosine_recall@5'],
        results_base_model['test-ir-eval_cosine_ndcg@10'],
        results_base_model['test-ir-eval_cosine_mrr@10']
    ],
    'QA-mpnet': [
        results_mpnet_model['test-ir-eval_cosine_accuracy@1'],
        results_mpnet_model['test-ir-eval_cosine_accuracy@5'],
        results_mpnet_model['test-ir-eval_cosine_precision@1'],
        results_mpnet_model['test-ir-eval_cosine_recall@1'],
        results_mpnet_model['test-ir-eval_cosine_recall@5'],
        results_mpnet_model['test-ir-eval_cosine_ndcg@10'],
        results_mpnet_model['test-ir-eval_cosine_mrr@10']
    ],
    'All-mpnet-base-v2': [
        results_all_mpnet_model['test-ir-eval_cosine_accuracy@1'],
        results_all_mpnet_model['test-ir-eval_cosine_accuracy@5'],
        results_all_mpnet_model['test-ir-eval_cosine_precision@1'],
        results_all_mpnet_model['test-ir-eval_cosine_recall@1'],
        results_all_mpnet_model['test-ir-eval_cosine_recall@5'],
        results_all_mpnet_model['test-ir-eval_cosine_ndcg@10'],
        results_all_mpnet_model['test-ir-eval_cosine_mrr@10']
    ]
}

# Define the column names (metrics)
columns = [
    'Accuracy@1', 'Accuracy@5', 'Precision@1', 'Recall@1', 'Recall@5', 'NDCG@10', 'MRR@10'
]

# Create the DataFrame and transpose it
df = pd.DataFrame(data, index=columns).T

# Format the DataFrame values to four decimal places
df = df.astype(float).map(lambda x: f'{x:.4f}')
df

In [ ]:
df.to_latex('../images/performance_metrics_table.tex', caption='Performance Metrics for Each Embedding Model', label='tab:performance_metrics', float_format="%.4f")



In [ ]:
num_documents = len(unique_corpus)
num_queries_total = len(queries)
num_queries_train = len(train_queries)
num_queries_val = len(val_queries)
num_queries_test = len(test_queries)

hyperparameters = {
    'per_gpu_batch_size': int(best_hyperparameters['per_gpu_batch_size']),
    'weight_decay': best_hyperparameters['weight_decay'],
    'learning_rate': best_hyperparameters['learning_rate'],
    'warmup_steps': int(best_hyperparameters['warmup_steps']),
    'num_epochs': int(best_hyperparameters['num_epochs'])
}

performance_metrics = {
    'Accuracy@1': results_best_model['test-ir-eval_cosine_accuracy@1'],
    'Accuracy@5': results_best_model['test-ir-eval_cosine_accuracy@5'],
    'Precision@1': results_best_model['test-ir-eval_cosine_precision@1'],
    'Recall@1': results_best_model['test-ir-eval_cosine_recall@1'],
    'Recall@5': results_best_model['test-ir-eval_cosine_recall@5'],
    'NDCG@10': results_best_model['test-ir-eval_cosine_ndcg@10'],
    'MRR@10': results_best_model['test-ir-eval_cosine_mrr@10']
}

# Combine all data into a single string for text file export
data = {
    'Number of Documents': num_documents,
    'Number of Queries (Total)': num_queries_total,
    'Number of Queries (Training)': num_queries_train,
    'Number of Queries (Validation)': num_queries_val,
    'Number of Queries (Test)': num_queries_test,
    'Best Hyperparameters': hyperparameters,
    'Performance Metrics': performance_metrics
}

def format_dict(data):
    if isinstance(data, dict):
        return '\n'.join([f"{key}: {value}" for key, value in data.items()])
    return str(data)

# Create the content for the text file
content = "\n\n".join([f"{key}:\n{format_dict(value)}" for key, value in data.items()])

# Define the text file name
text_file_path = '../docs/model_metrics.txt'

# Write the content to the text file
with open(text_file_path, 'w') as file:
    file.write(content)

print(f"Model metrics have been written to {text_file_path}")